# AWP Coding Agent

Generate, refactor, test, and document code projects with multi-agent delegation.
Configure in **Cell 1**, then **Run All**.

In [1]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  AWP CODING AGENT — EDIT THIS CELL, THEN RUN ALL                           ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ── 1. TASK ─────────────────────────────────────────────────────────────────────
# Describe the coding task. Be specific about language, structure, and deliverables.
TASK = (
    "Create a Python CLI tool called 'jsonq' that filters JSON files via JQ-like syntax. "
    "Requirements: "
    "1. CLI with argparse (input file, query expression, --pretty flag). "
    "2. Support dot notation (`.name`), array indexing (`[0]`), and wildcards (`[*]`). "
    "3. Comprehensive pytest tests (at least 10 test cases). "
    "4. A README.md with usage examples. "
    "5. Save all files to the output directory with proper project structure. "
    "6. Verify all tests pass by running pytest via code.execute."
)

# ── 2. INPUTS ───────────────────────────────────────────────────────────────────
# Provide context: existing code, specs, examples, config.
import pandas as pd, numpy as np

INPUTS = {
    # Existing code to refactor/extend:
    # "existing_code": "/path/to/src/",
    #
    # Spec or requirements doc:
    # "spec": "Must support Python 3.10+, type hints on all public functions.",
    #
    # Example input/output for the tool:
    "example_input": '{"users": [{"name": "Alice", "age": 30}, {"name": "Bob", "age": 25}]}',
    "example_query": ".users[*].name",
    "expected_output": '["Alice", "Bob"]',
    #
    # Project config:
    "project_config": {
        "language": "python",
        "python_version": "3.10+",
        "style": "ruff format, type hints, docstrings",
        "test_framework": "pytest",
    },
}

# ── 3. MODEL ────────────────────────────────────────────────────────────────────
import os
MODEL        = "openrouter/openai/gpt-4o-nano"   # Best for code gen
WORKER_MODEL = None

# ── 4. SECRETS ──────────────────────────────────────────────────────────────────
SECRETS = {
    # "GITHUB_TOKEN": os.getenv("GITHUB_TOKEN", ""),
    # "NPM_TOKEN": os.getenv("NPM_TOKEN", ""),
}

# ── 5. SKILLS ───────────────────────────────────────────────────────────────────
# Coding standards, architecture docs, API references.
SKILLS = [
    # "skills/python_coding_standards.md",
    # "skills/project_architecture/",
]

# ── 6. EXTERNAL TOOLS ──────────────────────────────────────────────────────────
EXTERNAL_TOOLS = []

# ── 7. BUDGET ───────────────────────────────────────────────────────────────────
MAX_LOOPS      = 100
MAX_TOKENS     = 1_000_000
MAX_WALLTIME   = 3000
MAX_TOOL_CALLS = 200         # Coding needs more tool calls (write, test, fix)
MAX_WORKERS    = 100
MAX_DEPTH      = 10

# ── 8. SANDBOX & PACKAGES ──────────────────────────────────────────────────────
SANDBOX  = "subprocess"
PACKAGES = ["pytest"]        # Always need a test runner
# PACKAGES += ["flask", "fastapi", "sqlalchemy", "pydantic"]

# ── 9. WORKER CAPABILITIES ─────────────────────────────────────────────────────
CODE_MODE     = True
TOOL_CREATION = True
VERBOSE       = True

# ── 10. TOOLS ───────────────────────────────────────────────────────────────────
TOOLS = [
    "code.execute", "file.read", "file.write", "file.list", "file.delete",
    "shell.execute", "web.search", "http.request",
    "arithmetic.add", "arithmetic.subtract", "arithmetic.multiply", "arithmetic.divide",
    "memory.read", "memory.write",
]
FORBIDDEN_TOOLS = []

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  END OF CONFIGURATION                                                      ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

In [2]:
# ── Setup & Execute ─────────────────────────────────────────
import os, time, shutil, subprocess, sys
from pathlib import Path
from dotenv import load_dotenv

try:
    import awp
except ImportError:
    print("Installing awp-agents from PyPI...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "awp-agents[data]", "--quiet"])
    import awp

for env_path in [Path.home() / "projects" / "awp" / ".env", Path(".env")]:
    if env_path.exists():
        load_dotenv(env_path)
        break

if "OPENROUTER_API_KEY" not in os.environ:
    raise RuntimeError("OPENROUTER_API_KEY not found. Create a .env file with it.")
os.environ["LLM_API_KEY"] = os.environ["OPENROUTER_API_KEY"]

from awp.data import AgentWorkflow, ExternalTool, ExternalToolSpec

OUTPUT_DIR = (Path.cwd() / "output_coding").resolve()
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

_secrets = {k: v for k, v in SECRETS.items() if v} or None

print(f"Model:      {MODEL}")
print(f"Task:       {TASK[:80]}...")
print(f"Inputs:     {list(INPUTS.keys())}")
print(f"Packages:   {PACKAGES}")
print(f"Budget:     loops={MAX_LOOPS}, tokens={MAX_TOKENS:,}, wall={MAX_WALLTIME}s")
print(f"Output:     {OUTPUT_DIR}")
print()

t0 = time.time()

result = AgentWorkflow(
    inputs=INPUTS,
    task=TASK,
    model=MODEL,
    worker_model=WORKER_MODEL,
    max_loops=MAX_LOOPS,
    max_total_tokens=MAX_TOKENS,
    max_wall_time=MAX_WALLTIME,
    max_tool_calls=MAX_TOOL_CALLS,
    max_total_workers=MAX_WORKERS,
    max_depth=MAX_DEPTH,
    sandbox=SANDBOX,
    packages=PACKAGES,
    code_mode=CODE_MODE,
    tool_creation=TOOL_CREATION,
    tools=TOOLS,
    forbidden_tools=FORBIDDEN_TOOLS,
    secrets=_secrets,
    skills=SKILLS or None,
    external_tools=EXTERNAL_TOOLS or None,
    output_dir=str(OUTPUT_DIR),
    verbose=VERBOSE,
).run()

elapsed = time.time() - t0
print(f"\nDone in {elapsed:.1f}s — Status: {result['status']}")

INFO:awp.data.workflow:Preparing inputs in workspace: /home/shumway/projects/agent-workflow-protocol/examples/jupyter/output_coding
INFO:awp.data.inputs:Input 'project_config': Dict -> /home/shumway/projects/agent-workflow-protocol/examples/jupyter/output_coding/workspace/inputs/project_config.json
INFO:awp.runtime.executor_factory:Creating subprocess executor (packages=['pytest'])


Model:      openrouter/openai/gpt-4o-nano
Task:       Create a Python CLI tool called 'jsonq' that filters JSON files via JQ-like synt...
Inputs:     ['example_input', 'example_query', 'expected_output', 'project_config']
Packages:   ['pytest']
Budget:     loops=100, tokens=1,000,000, wall=3000s
Output:     /home/shumway/projects/agent-workflow-protocol/examples/jupyter/output_coding



INFO:awp.data.workflow:Starting delegation loop: task=Create a Python CLI tool called 'jsonq' that filters JSON files via JQ-like synt
INFO:awp.runtime.delegation_loop_runner:DelegationLoop [2026-03-29_17-30-47_dfe20fde] depth=0 starting: Create a Python CLI tool called 'jsonq' that filters JSON files via JQ-like synt
INFO:awp.runtime.delegation_loop_runner:=== Iteration 1 ===
DEBUG:awp.runtime.llm:LLM request: model=openai/gpt-4o-nano, messages=2, tools=0
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 400 Bad Request"
DEBUG:awp.runtime.llm:LLM request: model=gpt-5-nano, messages=2, tools=0
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:awp.runtime.delegation_loop_runner:  Spawning worker: jsonq_builder
INFO:awp.runtime.delegation_loop_runner:Worker jsonq_builder: temperature=0.20 (from default)
DEBUG:awp.runtime.delegation_loop_runner:Worker jsonq_builder envelope:
{
  "worker_id": "jsonq_builde


  AWP DELEGATION LOOP DEBUG REPORT
  Model:         openrouter/openai/gpt-4o-nano
  Worker model:  openrouter/openai/gpt-4o-nano
  Budget:        loops=100, workers=100, tokens=1,000,000, wall_time=3000s, depth=10

  ────────────────────────────────────────────────────────
  Iteration 001
  ────────────────────────────────────────────────────────
    ────────────────────────────────────────────────
    MANAGER DECISION
    ────────────────────────────────────────────────
    Decision:    delegate
    Reasoning:
      | You asked to build a Python CLI tool 'jsonq' with tests, README, and project structure saved to the output directory. I'll create a self-contained, testable Python package under the output directory, including core logic (parse/query), a CLI, comprehensive pytest tests (>=10 cases), and README with usage examples. Then I'll run pytest to verify all tests pass.
    Full Manager Decision JSON:
      {
        "decision": "delegate",
        "reasoning": "You asked to buil

## Results

In [3]:
meta = result["metadata"]
status = result["status"]
status_icon = "\u2705" if status == "complete" else "\u274c"

print(f"{status_icon} Status:      {status}")
print(f"   Loops:       {meta['loops']}")
print(f"   Wall time:   {meta['wall_time']:.1f}s")
print(f"   Workers:     {meta['workers_spawned']}")
print(f"   Tool calls:  {meta['tool_calls']}")
print(f"   Tokens:      {meta['tokens_used']:,}")
print(f"   Artifacts:   {len(result['artifacts'])} files")

r = result["result"]
if isinstance(r, dict):
    print(f"\n   Confidence:  {r.get('confidence', 'N/A')}")
    if "termination_reason" in r:
        print(f"   Terminated:  {r['termination_reason']}")

✅ Status:      complete
   Loops:       3
   Wall time:   315.4s
   Workers:     2
   Tool calls:  8
   Tokens:      0
   Artifacts:   0 files

   Confidence:  0.92


## Generated Code

In [4]:
from IPython.display import display, Markdown

code_extensions = {".py", ".js", ".ts", ".go", ".rs", ".java", ".rb", ".sh", ".yaml", ".toml", ".cfg"}
output_path = OUTPUT_DIR / "output"

if output_path.exists():
    code_files = sorted(
        f for f in output_path.rglob("*")
        if f.is_file() and f.suffix in code_extensions
    )
    print(f"Generated {len(code_files)} code file(s):\n")
    for f in code_files:
        rel = f.relative_to(OUTPUT_DIR)
        content = f.read_text(encoding="utf-8", errors="replace")
        lang = f.suffix.lstrip(".")
        if lang == "py": lang = "python"
        display(Markdown(f"### `{rel}` ({len(content)} chars)\n\n```{lang}\n{content}\n```"))
else:
    print("No output directory found.")

Generated 0 code file(s):



## Documentation

In [5]:
output_path = OUTPUT_DIR / "output"
md_files = sorted(output_path.rglob("*.md")) if output_path.exists() else []
txt_files = sorted(output_path.rglob("*.txt")) if output_path.exists() else []

for md_file in md_files:
    rel = md_file.relative_to(OUTPUT_DIR)
    content = md_file.read_text(encoding="utf-8")
    display(Markdown(f"---\n**File: `{rel}`**\n\n{content}"))

for txt_file in txt_files:
    rel = txt_file.relative_to(OUTPUT_DIR)
    content = txt_file.read_text(encoding="utf-8")
    print(f"--- {rel} ---")
    print(content)

if not md_files and not txt_files:
    print("No documentation files generated.")

No documentation files generated.


## Delegation Graph

In [6]:
from IPython.display import display, HTML

runs_dir = OUTPUT_DIR / "workspace" / "runs"
if not runs_dir.exists():
    print("No run logs found.")
else:
    run_dir = sorted(runs_dir.iterdir())[-1]
    graph_html = run_dir / "execution_graph.html"

    if not graph_html.exists():
        try:
            from awp.runtime.execution_graph import generate_execution_graph
            generate_execution_graph(run_dir=run_dir, output_path=graph_html)
        except Exception as exc:
            print(f"Cannot generate execution graph: {exc}")

    if graph_html.exists():
        html_content = graph_html.read_text(encoding="utf-8")
        import html as html_mod
        escaped = html_mod.escape(html_content)
        display(HTML(
            f'<iframe srcdoc="{escaped}" width="100%" height="700" '
            f'style="border:1px solid #333; border-radius:8px;"></iframe>'
        ))
    else:
        print(f"No execution graph available for {run_dir.name}")

/home/shumway/projects/agent-workflow-protocol/.venv/lib/python3.12/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


## All Artifacts

In [7]:
output_path = OUTPUT_DIR / "output"
if output_path.exists():
    total = 0
    files = sorted(output_path.rglob("*"))
    for f in files:
        if f.is_file():
            sz = f.stat().st_size
            total += sz
            rel = str(f.relative_to(OUTPUT_DIR))
            marker = "\u2705" if sz > 100 else "\u274c"
            print(f"  {marker} {rel:<50s} {sz:>8,} bytes")
    print(f"\n  Total: {total:,} bytes in {sum(1 for f in files if f.is_file())} files")
else:
    print("No output directory found.")


  Total: 0 bytes in 0 files
